# Phase 2: planner baselines (greedy, self-consistency, oracle best-of-N)

Run **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all** (or run cell by cell).
It clones the public repo, makes the problems, runs the LLM, checks every plan with the Python simulator,
writes a summary table, and downloads a zip. Free Colab can disconnect: results are saved to your Google Drive
as they are produced, and re-running the same cells **resumes** where it stopped.

## 0. Settings (edit only this cell)

In [ ]:
MODEL      = "Qwen/Qwen2.5-7B-Instruct"   # 3B was too weak; 7B in 4-bit works
LOAD_4BIT  = True                           # needed for 7B on a T4
STYLE      = "state"                        # "state" = write the towers after each action (v2); "plain" = just the plan (v1)
SIZES      = "3 4 5"                        # blocks per problem (6 was never solved)
COUNT      = 20                             # problems per size
N_SAMPLES  = 16                             # samples per problem
MAX_NEW    = 450                            # max generated tokens per sample (state style needs more)
LIMIT      = 0                              # 0 = all problems; set 6 for a quick test
USE_DRIVE  = True                           # keep results in Google Drive so a disconnect loses nothing
BUILD_SOLVER = False                        # True = also build Fast Downward + VAL (about 15 min, optional)

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import os, subprocess
REPO = "https://github.com/Manidip-Debnath/llm-planning-process-verifier.git"
if not os.path.exists("/content/llm-planning-process-verifier"):
    !git clone -q {REPO} /content/llm-planning-process-verifier
%cd /content/llm-planning-process-verifier
!git pull -q
!git log --oneline | head -3

NAME = MODEL.split("/")[-1] + ("_state" if STYLE == "state" else "")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNDIR = "/content/drive/MyDrive/planning_prm_runs"
else:
    RUNDIR = "/content/runs"
os.makedirs(RUNDIR, exist_ok=True)
OUT = f"{RUNDIR}/{NAME}.jsonl"
print("results file:", OUT)

## 2. Generate the problems (seeded, identical to your PC)

In [ ]:
!python data/gen_blocksworld.py --sizes {SIZES} --count {COUNT} --seed 0 --out data/blocksworld/problems
!ls data/blocksworld/problems | wc -l

## 3. Quick check (3 problems). Read the model output before the long run.

In [ ]:
LIM = "--limit 3 --n-samples 4"
Q4 = "--load-4bit" if LOAD_4BIT else ""
!python planners/run_baselines.py --backend hf --model {MODEL} {Q4} --style {STYLE} --max-new-tokens {MAX_NEW} --problems data/blocksworld/problems {LIM} --out /content/quick.jsonl

In [ ]:
import json
r = json.loads(open("/content/quick.jsonl").readline())
print("GREEDY OUTPUT:\n", r["greedy"]["text"])
print("tokens:", r["greedy"]["tokens"])
print("parsed plan:", r["greedy"]["plan"])
print("valid:", r["greedy"]["valid"], "| reason:", r["greedy"]["reason"])

## 4. Full baseline run (resumable)

With step-by-step output each problem takes roughly 45-90 s on a T4, so 60 problems take about 1-1.5 hours. If the session disconnects, reconnect,
run sections 0, 1, 2 again, then this cell: it skips problems already saved in Drive.

In [ ]:
LIM = f"--limit {LIMIT}" if LIMIT else ""
Q4 = "--load-4bit" if LOAD_4BIT else ""
!python planners/run_baselines.py --backend hf --model {MODEL} {Q4} --style {STYLE} --max-new-tokens {MAX_NEW} --problems data/blocksworld/problems --n-samples {N_SAMPLES} {LIM} --out {OUT}

## 5. Optional: Fast Downward + VAL (set BUILD_SOLVER = True in section 0)

Gives optimal plan costs (for the optimality gap) and checks that the Python simulator agrees with VAL.

In [ ]:
if BUILD_SOLVER:
    !git clone -q --depth 1 https://github.com/aibasel/downward.git /content/fast-downward
    !cd /content/fast-downward && python3 build.py -j2 > /content/fd_build.log 2>&1; tail -2 /content/fd_build.log
    !git clone -q --depth 1 https://github.com/KCL-Planning/VAL.git /content/VAL
    !mkdir -p /content/VAL/build && cd /content/VAL/build && cmake .. -DCMAKE_POLICY_VERSION_MINIMUM=3.5 > /content/val_cmake.log 2>&1 && make -j2 > /content/val_build.log 2>&1; ls -l /content/VAL/build/bin/Validate
    os.environ["FD_PATH"] = "/content/fast-downward/fast-downward.py"
    os.environ["VAL_PATH"] = "/content/VAL/build/bin/Validate"
    !python data/solve_optimal.py --domain data/domains/blocksworld/domain.pddl --problems data/blocksworld/problems --out data/blocksworld/solutions.jsonl
    !python eval/check_sim_vs_val.py --domain data/domains/blocksworld/domain.pddl --problems data/blocksworld/problems --solutions data/blocksworld/solutions.jsonl
else:
    print("skipped (BUILD_SOLVER = False)")

## 6. Summary tables

In [ ]:
os.makedirs("results", exist_ok=True)
SOL = "--solutions data/blocksworld/solutions.jsonl" if os.path.exists("data/blocksworld/solutions.jsonl") else ""
!python eval/summarize_baselines.py --results {OUT} {SOL} --out results/baselines_{NAME}.md

## 7. Download

Put `results/baselines_*.md` in your project's `results/` folder and commit it. The raw `.jsonl` stays out of Git
(it is also in your Drive folder `planning_prm_runs`).

In [ ]:
!cp {OUT} /content/ && cd /content/llm-planning-process-verifier && zip -q -r /content/results_download.zip results && zip -q -j /content/results_download.zip {OUT}
from google.colab import files
files.download("/content/results_download.zip")